In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.ProcessKnowledgeGraph import ProcessKnowledgeGraph
from src.utils import *
from src.KnowledgeImporter import SimpleEventLogImporter, Keys

In [88]:
from src.utils import BASE_PROCESS_ONTOLOGY as BPO

In [3]:
import importlib.resources

In [4]:
draw_graph(ProcessKnowledgeGraph())

GraphWidget(layout=Layout(height='800px', width='100%'))

# Load Event Log Information

In [5]:
from pandas.api.types import is_string_dtype, is_numeric_dtype, is_datetime64_any_dtype

In [6]:
import pandas as pd
import pm4py
log = pm4py.read_xes('../Sepsis Cases - Event Log.xes')


/home/lblinux/.local/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

### Extract Key Entities

In [7]:
from rdflib import XSD

In [8]:
# Manual annotation hints
entity_columns = []
value_columns = []
ignore_columns = [Keys.CASE, 'time:timestamp', 'lifecycle:transition']

In [9]:
pkg = ProcessKnowledgeGraph()
pkg.parse('./src/base_ontology.ttl')

importer = SimpleEventLogImporter(pkg, entity_columns=entity_columns, value_columns=value_columns, ignore_columns=ignore_columns)
importer.import_event_log_entities(log)
#draw_graph(importer.addition_graph)

InfectionSuspected, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
org:group, object : ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' '?' 'I']
=> Entity column
Added type owl class for org:group: http://example.org/type_org%3Agroup
DiagnosticBlood, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
DisfuncOrg, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
SIRSCritTachypnea, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
Hypotensie, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
SIRSCritHeartRate, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
Infusion, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
DiagnosticArtAstrup, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema

In [45]:
importer.ui.visualize_addition_graph()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [11]:
importer.load()
draw_graph(pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [12]:
from src.KnowledgeImporter import TextualImporter
x = TextualImporter(pkg)
# x.import_content_from_statement('The activity CRP describes that a blood test for measuring CRP values is performed')
x.import_content_from_statement('The process value CRP represents the mg of C-reactive protein per liter of blood in a blood test')
x.import_content_from_statement('The process value Lactic Acid measures the amount of lactic acid in a blood test')
x.import_content_from_statement('The process value Leucocytes represents the number of white blood cells in a blood test')
x.import_content_from_statement('The process value Hypoxie defines whether hypoxia has been detected for the patient')
x.visualize_addition_graph()

```turtle
log:processValue_CRP a :ProcessValue ;
    rdfs:comment "The process value CRP represents the mg of C-reactive protein per liter of blood in a blood test." .
```
```turtle
log:processValue_LacticAcid a :ProcessValue ;
    rdfs:label "Lactic Acid" ;
    rdfs:comment "Measures the amount of lactic acid in a blood test." .
```
```turtle
log:processValue_Leucocytes a :ProcessValue ;
    rdfs:comment "The number of white blood cells in a blood test." .
```
```turtle
log:processValue_Hypoxie a :ProcessValue ;
    rdfs:label "Hypoxie" ;
    :dataType xsd:boolean ;
    rdfs:comment "Defines whether hypoxia has been detected for the patient." .
```


GraphWidget(layout=Layout(height='620px', width='100%'))

In [13]:
x.ui.show_validation()

Output()

# Ontology Extraction

In [14]:
sepon = Graph()
sepon.bind('sepon', Namespace('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology'), override=True)
sepon.parse(importlib.resources.path('example_domains.sepsis', 'SEPON.ttl'))

<Graph identifier=N67669810365b447bb167e7f12b553d34 (<class 'rdflib.graph.Graph'>)>

In [15]:
query = '''
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sepon: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#>

SELECT ?subject ?predicate ?object WHERE {
  {
    SELECT DISTINCT ?node WHERE {
      {
        ?node rdf:type sepon:Diagnostic_Biomarker .
      }
      UNION
      {
        ?node rdfs:subClassOf+ sepon:Diagnostic_Biomarker .
      }
    }
  }

  # Edges either going out from or coming into ?node
  {
    ?node ?predicate ?object .
    BIND(?node AS ?subject)
  }
  UNION
  {
    ?subject ?predicate ?node .
    BIND(?node AS ?object)
  }
}
'''

In [16]:
from src.KnowledgeImporter import ExistingOntologyImporter
ontology_importer = ExistingOntologyImporter(pkg)
ontology_importer.import_ontology(sepon, query)

Output()

In [470]:
len(ontology_importer.addition_graph)

1962

In [471]:
len(sepon)

12942

# Alignment

In [18]:
diagnoses = set(pkg.subjects(RDF.type, pkg.namespace_manager.expand_curie('log:type_Diagnose')))
diagnosis_filter = lambda node : node not in diagnoses
llm_approved = ontology_importer.align(target_node_filter=diagnosis_filter, addition_text_params={'blacklist' : {URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#reference')}})
llm_approved

Textualizing graphs for alignment...
Calculating basic alignment...
Trial by LLM...
ns1:C-Reactive_Protein -> log:processValue_CRP
ns1:Blood_Cell_Count -> log:processValue_Leucocytes
ns1:Leukocyte_Count -> log:processValue_Leucocytes
ns1:Blood_Chemistry_Measurement -> log:processValue_Leucocytes
ns1:Blood_Chemistry_Measurement -> log:processValue_CRP
ns1:Blood_Chemistry_Measurement -> log:processValue_LacticAcid
ns1:Lactate -> log:processValue_LacticAcid


{(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'),
  rdflib.term.URIRef('http://example.org/processValue_Leucocytes')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_CRP')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_LacticAcid')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_Leucocytes')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'),
  rdflib.term.URIRef('http://example.org/processValue_CRP')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOnt

In [43]:
ontology_importer.ui.show_alignment_validation(llm_approved)

Output()

# ===============================================================

# Reasoning Workbench

In [116]:
log.loc[0]

InfectionSuspected                                True
org:group                                            A
DiagnosticBlood                                   True
DisfuncOrg                                        True
SIRSCritTachypnea                                 True
Hypotensie                                        True
SIRSCritHeartRate                                 True
Infusion                                          True
DiagnosticArtAstrup                               True
concept:name                           ER Registration
Age                                               85.0
DiagnosticIC                                      True
DiagnosticSputum                                 False
DiagnosticLiquor                                 False
DiagnosticOther                                  False
SIRSCriteria2OrMore                               True
DiagnosticXthorax                                 True
SIRSCritTemperature                               True
time:times

In [119]:
case_attributes = list(log.groupby('case:concept:name').agg('nunique').max().where(lambda x : x == 1).dropna().index)
case_attributes

['InfectionSuspected',
 'DiagnosticBlood',
 'DisfuncOrg',
 'SIRSCritTachypnea',
 'Hypotensie',
 'SIRSCritHeartRate',
 'Infusion',
 'DiagnosticArtAstrup',
 'Age',
 'DiagnosticIC',
 'DiagnosticSputum',
 'DiagnosticLiquor',
 'DiagnosticOther',
 'SIRSCriteria2OrMore',
 'DiagnosticXthorax',
 'SIRSCritTemperature',
 'DiagnosticUrinaryCulture',
 'SIRSCritLeucos',
 'Oligurie',
 'DiagnosticLacticAcid',
 'lifecycle:transition',
 'Diagnose',
 'Hypoxie',
 'DiagnosticUrinarySediment',
 'DiagnosticECG']

In [146]:
from src.KnowledgeImporter import OnlineEventImporter
event_interface = OnlineEventImporter(pkg, case_attributes=case_attributes, ignore_columns={'lifecycle:transition'})
data = log[log['case:concept:name'] == 'B']
event_interface.translate_event(data.iloc[0])
#draw_graph(event_interface.addition_graph)
for index, event in data.iterrows():
    event_interface.translate_event(event)

In [147]:
draw_graph(event_interface.addition_graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [148]:
event_interface.ui.visualize_addition_graph()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [124]:
event_interface.attribute_aliases

{'concept:name': rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/Activity'),
 'case:concept:name': rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/Case'),
 'org:resource': rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/Resource')}

# ===============================================================

### Extract "Rules"

In [37]:
from pm4py import discover_declare, discover_log_skeleton

In [38]:
declare = discover_declare(log, min_support_ratio=0.98, min_confidence_ratio=0.98)

In [39]:
declare

{'responded_existence': {('ER Registration',
   'ER Sepsis Triage'): {'support': 1050, 'confidence': 1049},
  ('ER Sepsis Triage', 'ER Registration'): {'support': 1049,
   'confidence': 1049},
  ('ER Registration', 'ER Triage'): {'support': 1050, 'confidence': 1050},
  ('ER Sepsis Triage', 'ER Triage'): {'support': 1049, 'confidence': 1049},
  ('ER Triage', 'ER Sepsis Triage'): {'support': 1050, 'confidence': 1049},
  ('ER Triage', 'ER Registration'): {'support': 1050, 'confidence': 1050}},
 'existence': {'ER Sepsis Triage': {'support': 1050, 'confidence': 1049},
  'ER Registration': {'support': 1050, 'confidence': 1050},
  'ER Triage': {'support': 1050, 'confidence': 1050}},
 'exactly_one': {'ER Sepsis Triage': {'support': 1049, 'confidence': 1049},
  'ER Registration': {'support': 1050, 'confidence': 1050},
  'ER Triage': {'support': 1050, 'confidence': 1047}},
 'altprecedence': {('ER Registration', 'ER Sepsis Triage'): {'support': 1049,
   'confidence': 1042},
  ('ER Registration', 

In [155]:
discover_log_skeleton(log)

{'equivalence': {('ER Sepsis Triage', 'ER Registration'),
  ('IV Antibiotics', 'ER Registration'),
  ('IV Antibiotics', 'ER Sepsis Triage'),
  ('IV Liquid', 'ER Registration'),
  ('IV Liquid', 'ER Sepsis Triage'),
  ('IV Liquid', 'IV Antibiotics'),
  ('Release A', 'ER Registration'),
  ('Release B', 'ER Registration'),
  ('Release B', 'ER Sepsis Triage'),
  ('Release B', 'ER Triage'),
  ('Release C', 'ER Registration'),
  ('Release C', 'ER Sepsis Triage'),
  ('Release C', 'ER Triage'),
  ('Release D', 'ER Registration'),
  ('Release D', 'ER Sepsis Triage'),
  ('Release D', 'ER Triage'),
  ('Release E', 'ER Registration'),
  ('Release E', 'ER Sepsis Triage'),
  ('Release E', 'ER Triage'),
  ('Return ER', 'ER Registration')},
 'always_after': {('Release B', 'Admission NC'),
  ('Release C', 'Return ER'),
  ('Release D', 'Return ER'),
  ('Release E', 'Return ER')},
 'always_before': set(),
 'never_together': {('Release A', 'Release B'),
  ('Release A', 'Release C'),
  ('Release A', 'Releas

# Ontology Extraction

In [7]:
from src.utils import *

In [9]:
def nodes_in_dist(graph, nodes, dist, filter_func=lambda x: not x.startswith(OWL) and not x.startswith(RDF) and not x.startswith(RDFS)):
    triples = set()
    for node in nodes:
        triples |= set(graph.triples((node, None, None)))
        triples |= set(graph.triples((None, None, node)))
    if dist > 1:
        neighborhood = set([x[0] for x in triples]) | set([x[2] for x in triples]) 
        neighborhood = set(filter(filter_func, neighborhood)) 
        print(neighborhood)
        triples |= nodes_in_dist(graph, neighborhood, dist-1)
    return triples

In [10]:
URIRef('http://www.w3.org/2002/07/owl#Class').startswith(OWL)

True

In [11]:
OWL

Namespace('http://www.w3.org/2002/07/owl#')

In [12]:
lc_node = URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')
g2 = Graph()
copy_namespaces(g2, sepon)
g2 += nodes_in_dist(sepon, [lc_node], 2)
#g2 += list(sepon.triples((x, Path(), None)))
#g2 += list(sepon.triples((None, Path(), x)))
draw_graph(g2)

{rdflib.term.Literal('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C51948'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neonatal_Sepsis_Biomarker'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.Literal('Leukocyte Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Granulocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neutrophil_Count'), rdflib.term.Literal('White Blood Cell Count\nWhite Cell Count\nWhite Blood Cells\nLeukocyte Count\nLeukocytes\nWBC'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Eosinophil_Count'), rdflib.term.Literal('A test to determine the number of leukocytes in a biospecimen.'), rdflib.term.URI

GraphWidget(layout=Layout(height='800px', width='100%'))

In [13]:
x = URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Procedure')
g2 = Graph()
copy_namespaces(g2, sepon)
g2 += nodes_in_dist(sepon, sepon.subjects(object=x, predicate=RDFS.subClassOf * '*'), 1)
#g2 += list(sepon.triples((x, Path(), None)))
#g2 += list(sepon.triples((None, Path(), x)))
# draw_graph(g2)
list()

[]

In [14]:
len(g2)

3215

In [15]:
len(sepon)

12942

In [20]:
log.dtypes

InfectionSuspected                        object
org:group                                 object
DiagnosticBlood                           object
DisfuncOrg                                object
SIRSCritTachypnea                         object
Hypotensie                                object
SIRSCritHeartRate                         object
Infusion                                  object
DiagnosticArtAstrup                       object
concept:name                              object
Age                                      float64
DiagnosticIC                              object
DiagnosticSputum                          object
DiagnosticLiquor                          object
DiagnosticOther                           object
SIRSCriteria2OrMore                       object
DiagnosticXthorax                         object
SIRSCritTemperature                       object
time:timestamp               datetime64[ns, UTC]
DiagnosticUrinaryCulture                  object
SIRSCritLeucos      

In [22]:
from rdflib import OWL

In [23]:
from rdflib import Literal
Literal(5).datatype

rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')

rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label')

In [249]:
pkg.namespace['foo']

rdflib.term.URIRef('http://example.org/foo')

### Matching With Ontology

In [26]:
list(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))[0].split('/')[-1]

'InfectionSuspected'

In [28]:
ontology_nodes = set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))
context_ids = []
contexts = []
for node in ontology_nodes:
    g = Graph()
    g += context(node, sepon)
    _context = '\n\n'.join(g.serialize(format='ttl').split('.\n\n')[1:]).strip() # Remove namespaces
    context_ids.append(node)
    contexts.append(_context)
    # display(list(context(node, sepon)))

[print(f'\n====\n\n{x}\n{y}') for x,y in zip(context_ids, contexts)]


====

http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hyaluronan_Synthases
ns1:Hyaluronan_Synthases rdfs:label "Hyaluronan Synthases" ;
    ns1:Synonyms """Synthase 3, Hyaluronan
Synthetase, Hyaluronic Acid
Synthetase, Hyaluronan
HAS1 Protein
Synthase 2, Hyaluronan
Synthase, Hyaluronate
Hyaluronan Synthase 3
Hyaluronate Synthetase
Synthase 1, Hyaluronan
HAS2 Protein
Protein, HAS3
Hyaluronan Synthase
Hyaluronic Acid Synthetase
HAS3 Protein
3, Hyaluronan Synthase
Hyaluronan Synthetase
Hyaluronan Synthase 2
Synthase, Hyaluronan
Synthetase, Hyaluronate
Hyaluronate Synthase
hasA Enzyme
Synthases, Hyaluronan
Hyaluronan Synthase 1""" ;
    ns1:definition "Membrane-associated glucuronosyltransferases that catalyze the reaction of UDP-N-acetyl-D-glucosamine and UDP-D-glucuronate to produce HYALURONAN. HYALURONAN SYNTHASE 2 (HAS2) is essential for embryogenesis and its expression by tumor cells is associated with metastasis." ;
    rdfs:subClassOf ns1:Adult_Sepsis_Diagnostic

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

SIRSCritHeartRate -> sepon:Heart_Rate is not a match, one is the parameter, the other is a boolean on that parameter

In [30]:
matches

[(rdflib.term.URIRef('http://example.org/instances/process_values/CRP'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/LacticAcid'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Leucocytes'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'))]

In [60]:
_matches = [(URIRef('http://example.org/instances/process_values/CRP'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (URIRef('http://example.org/instances/process_values/LacticAcid'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (URIRef('http://example.org/instances/process_values/Leucocytes'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'))]

In [61]:
manually_added_matchings = [
    (importer.entity_instance_node('process_value', 'Hypoxie'), URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'))    
]

In [65]:
matches.extend(manually_added_matchings)
matches

[(rdflib.term.URIRef('http://example.org/instances/process_values/CRP'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/LacticAcid'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Leucocytes'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Hypoxie'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'))]

In [66]:
def rename_identifier(g: Graph, old_uri: URIRef, new_uri: URIRef):
    # Collect all affected triples
    to_add = []
    to_remove = []
    
    for s, p, o in g:
        if s == old_uri or o == old_uri:
            new_s = new_uri if s == old_uri else s
            new_o = new_uri if o == old_uri else o
            to_remove.append((s, p, o))
            to_add.append((new_s, p, new_o))
    
    # Apply changes
    for triple in to_remove:
        g.remove(triple)
    for triple in to_add:
        g.add(triple)

In [70]:
for old, new in matches:
    rename_identifier(pkg, old, new)
    pkg += context(new, sepon)
    print(f'Replaced {old}\nwith {new}')

Replaced http://example.org/instances/process_values/CRP
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein
Replaced http://example.org/instances/process_values/LacticAcid
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate
Replaced http://example.org/instances/process_values/Leucocytes
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count
Replaced http://example.org/instances/process_values/Hypoxie
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia


In [88]:
pkg.bind('sepon', 'http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#')

In [35]:
#TODO memorize this translation

In [69]:
list(context(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'), sepon))

[(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Ventilation_or_Perfusion_Signs_and_Symptoms')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Synonyms'),
  rdflib.term.Literal('Hypoxia\nHypoxic\nhypoxia\nhypoxic')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#definition'),
  rdflib.term.Literal('A decrease in the amount of oxygen in the body. Symptoms range from mild (impaired judgment, memory loss, impaired motor coordination) to severe (seizures and coma).')),
 (rdflib.term.URIRef('htt

In [89]:
draw_graph(pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

#### Matching Workbench

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')
biomarker_embeddings = model.encode(contexts, convert_to_tensor=True)
process_nodes = set(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))
for process_node in process_nodes:
    process_node_label = process_node.split('/')[-1]
    name_embedding = model.encode(process_node_label, convert_to_tensor=True)
    
    # Compute cosine similarity with all biomarker contexts
    similarities = util.cos_sim(name_embedding, biomarker_embeddings)[0]
    
    # Get top 3 matches
    top_indices = similarities.argsort(descending=True)[:3]

    display(similarities)

process_nodes = set(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))
ontology_nodes = set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))

from difflib import SequenceMatcher

matches = {}
for process_node in process_nodes:
    process_node_label = process_node.split('/')[-1]
    best_match = None
    best_score = 0
    for n2 in ontology_nodes:
        label2 = n2.split('#')[-1]
        score = SequenceMatcher(None, process_node_label, label2).ratio()
        if score > best_score:
            best_score = score
            best_match = (n2, label2)
    print(f"{process_node_label} -> {best_match[1]} (score: {best_score:.2f})")
    # confirm = input("Accept match? (y/n): ")
    #if confirm.lower() == 'y':
    if True:
        matches[process_node] = best_match[0]

matches

set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))

# Text Extraction

### Crawl the rules from the Clinical Guidelines

In [72]:
import requests
import re
from bs4 import BeautifulSoup

In [73]:
response = requests.get("https://www.sccm.org/clinical-resources/guidelines/guidelines/surviving-sepsis-guidelines-2021")
response.raise_for_status()
# print(response.text)

In [74]:
soup = BeautifulSoup(response.text, "html.parser")

In [75]:
guideline_container = soup.findAll("div", {"class": "col-main p-0"})[0]

In [76]:
reg = re.compile(r'\r\nQuality of evidence:.*')
rules = []
for guideline in guideline_container.findAll("div", {"class": "card mb-3"}):
    text = guideline.find('p', {'class' : 'mb-1'}).text
    text = reg.sub('', text)
    print(text)
    print('')
    rules.append(text)

For hospitals and health systems, we recommend using a performance improvement program for sepsis, including sepsis screening for acutely ill, high-risk patients and standard operating procedures for treatment.

We recommend against using qSOFA compared with SIRS, NEWS, or MEWS as a single screening tool for sepsis or septic shock.

For adults suspected of having sepsis, we suggest measuring blood lactate.

Sepsis and septic shock are medical emergencies, and we recommend that treatment and resuscitation begin immediately.

For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.

For adults with sepsis or septic shock, we suggest using dynamic measures to guide fluid resuscitation over physical examination or static parameters alone.

For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum lactate in patients with elevated lactate leve

In [77]:
rules

['For hospitals and health systems, we recommend using a performance improvement program for sepsis, including sepsis screening for acutely ill, high-risk patients and standard operating procedures for treatment.',
 'We recommend against using qSOFA compared with SIRS, NEWS, or MEWS as a single screening tool for sepsis or septic shock.',
 'For adults suspected of having sepsis, we suggest measuring blood lactate.',
 'Sepsis and septic shock are medical emergencies, and we recommend that treatment and resuscitation begin immediately.',
 'For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.',
 'For adults with sepsis or septic shock, we suggest using dynamic measures to guide fluid resuscitation over physical examination or static parameters alone.',
 'For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum\xa0lactate in patients wit

In [78]:
import json
cg_filename = './clinical_guidelines.json'

In [79]:
with open(cg_filename, 'w') as f:
    json.dump(rules, f)

In [80]:
with open(cg_filename) as f:
    assert rules == json.load(f)

In [81]:
with open(cg_filename) as f:
    rules = json.load(f)

### Alt: Curated Rules

In [82]:
curated_rules = [
    # From Surviving SEPSIS Guidelines
    'For adults with septic shock and severe metabolic acidemia (pH ≤ 7.2) and acute kidney injury (AKIN score 2 or 3), we suggest using sodium bicarbonate therapy.',
    'For adults with septic shock and hypoperfusion-induced lactic acidemia, we suggest against using sodium bicarbonate therapy to improve hemodynamics or to reduce vasopressor requirement',
    'For adults with sepsis or septic shock, we suggest against using IV vitamin C.',
    'For adults with sepsis or septic shock, we recommend initiating insulin therapy at a glucose level of ≥ 180mg/dL (10mmol/L).',
    'For adults with sepsis or septic shock, we recommend using low-molecular-weight heparin.',
    'For adults with sepsis or septic shock, we suggest against using IV immunoglobulin.',
    'For adults with septic shock and an ongoing requirement for vasopressor therapy, we suggest using IV corticosteroids.',
    'For adults with sepsis or septic shock who require ICU admission, we suggest admitting the patients to the ICU within 6 hours.',
    'For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.',
    # From Papers etc.
    '1. between ER Sepsis Triage and IV Antibiotics should be less than 1 hour',
    '2. between ER Sepsis Triage and LacticAcid should be less than 3 hours.'
    
]

### Translate the Rules

In [51]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [52]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

context_graph = Graph()
context_graph.parse('./src/base_ontology.ttl')
print(context_graph.serialize(format='ttl'))

In [55]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
'''Your task is to translate a textual rule into an equivalent SPARQL query based on a given ontology, so that whenever the rule is violated, the result set of the query is non-empty and vice versa. 
Create a query that includes the variable "?case", which relates to one specific process instance.
Please formulate a SELECT-WHERE query. 
Please only output the SPARQL query and nothing else. Don't output any notes or justifications.

For generating the rules, please consider the following contextual schema information in OWL-RDF format and key entities in RDF format.  

{context}''',
        ),
        ("human", "Please translate the following rule {rule}"),
    ]
)

prompt_context = context_graph.serialize(format='ttl') + '''

@prefix sepsis: <http://example.org/sepsis/> .

sepsis:Measurement rdf:type owl:Class ;
    rdfs:label "Measurement" ;    
    rdfs:comment "A measurement of a clinical parameter".

sepsis:Clinical_Parameter rdf:type owl:Class ;
    rdfs:label "Clinical Parameter" ;    
    rdfs:comment "A clinical parameter".

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:mes_value rdf:type owl:DatatypeProperty, owl:FunctionalProperty ;    
    rdfs:comment "The value of a measurement" ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range xsd:decimal .

sepsis:produced rdf:type owl:ObjectProperty, owl:FunctionalProperty ;    
    rdfs:comment "A measurement has been produced" ;
    rdfs:domain type:Task ;
    rdfs:range sepsis:Measurement .

sepsis:leucocyte_count a sepsis:Clinical_Parameter ;
    rdfs:label "Leucocyte Count" ;    
    rdfs:comment "The number of Leucocytes (white blood cells) in 10^9 per liter of blood.".
'''

In [115]:
prompt_context = pkg.serialize(format='ttl')
print('\n'.join(prompt_context.split('\n')[:100]))

@prefix Activity: <http://example.org/instances/Activitys/> .
@prefix Diagnose: <http://example.org/instances/Diagnoses/> .
@prefix lifecycle%3Atransition: <http://example.org/instances/lifecycle%3Atransitions/> .
@prefix ns1: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#> .
@prefix org%3Agroup: <http://example.org/instances/org%3Agroups/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix process_value: <http://example.org/instances/process_values/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

Activity:Admission%20IC a type:Activity .

Activity:Admission%20NC a type:Activity .

Activity:CRP a type:Activity ;
    relation:writes_value process_value:Age,
        ns1:C-Reactive_Protein .

Activity:ER%20Registration a type:Activity ;
    re

In [118]:
#TODO fix this bug, this is just a workaround
prompt_context = prompt_context.replace('ns1:', 'sepon:')

``` python
context = '''
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix sepsis: <http://example.org/sepsis/> .

type:Task rdf:type owl:Class ;
    rdfs:label "Task" ;
    rdfs:comment "One specific task to be performed".

type:Case rdf:type owl:Class ;
    rdfs:label "Case" ;
    rdfs:comment "One instance of a process such as one shopping order".

type:Activity rdf:type owl:Class ;
    rdfs:label "Activity" ;    
    rdfs:comment "A recurring activity of a process".

relation:instanceOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Activity .

relation:partOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Case .


sepsis:Measurement rdf:type owl:Class ;
    rdfs:label "Measurement" ;    
    rdfs:comment "A measurement of a clinical parameter".

sepsis:Clinical_Parameter rdf:type owl:Class ;
    rdfs:label "Clinical Parameter" ;    
    rdfs:comment "A clinical parameter".

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:mes_value rdf:type owl:DatatypeProperty, owl:FunctionalProperty ;    
    rdfs:comment "The value of a measurement" ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range xsd:decimal .

sepsis:produced rdf:type owl:ObjectProperty, owl:FunctionalProperty ;    
    rdfs:comment "A measurement has been produced" ;
    rdfs:domain type:Task ;
    rdfs:range sepsis:Measurement .

sepsis:leucocyte_count a sepsis:Clinical_Parameter ;
    rdfs:label "Leucocyte Count" ;    
    rdfs:comment "The number of Leucocytes (white blood cells) in 10^9 per liter of blood.".
    
'''
```

In [119]:
chain = prompt | llm

def translate_rule(rule):
     return chain.invoke(
        {
            "context": prompt_context,
            "rule": rule,
        }
    )

rule = "Leucocyte measurements must be between 4.0 and 12.0 million."
response = translate_rule(rule)
print(response.content)

```sparql
SELECT ?case WHERE {
    ?task relation:partOf ?case .
    ?task relation:performedBy ?resource .
    ?task instanceOf Activity:Leucocytes .
    ?value a sepon:Leukocyte_Count ;
           relation:data_type xsd:double .
    FILTER(?value < 4.0 || ?value > 12.0)
}
```


In [122]:
print(unwrap_markdown_code(response.content).replace(' instanceOf', ' relation:instanceOf'))

SELECT ?case WHERE {
    ?task relation:partOf ?case .
    ?task relation:performedBy ?resource .
    ?task relation:instanceOf Activity:Leucocytes .
    ?value a sepon:Leukocyte_Count ;
           relation:data_type xsd:double .
    FILTER(?value < 4.0 || ?value > 12.0)
}


In [112]:
pkg.query('''
SELECT ?case WHERE {
  ?task relation:partOf ?case .
  ?task relation:performedBy ?resource .
  ?task relation:instanceOf Activity:Leucocytes .
  ?value a sepo:Leukocyte_Count ;
         relation:data_type ?leukocyteCount .
  FILTER(?leukocyteCount < 4.0 || ?leukocyteCount > 12.0)
}
''')

In [125]:
list(pkg.query(unwrap_markdown_code(response.content.replace(' instanceOf', ' relation:instanceOf'))))

[]

In [102]:
printmd('<span style="color:red">ERROR</span>')

<span style="color:red">ERROR</span>

In [126]:
for rule in curated_rules:
    print(rule)
    response = translate_rule(rule)
    printmd(response.content)
    try:
        res = list(pkg.query(unwrap_markdown_code(response.content.replace(' instanceOf', ' relation:instanceOf'))))
        print(res)
    except Exception as e:
        printmd('<span style="color:red">ERROR</span>')
        print(e)

For adults with septic shock and severe metabolic acidemia (pH ≤ 7.2) and acute kidney injury (AKIN score 2 or 3), we suggest using sodium bicarbonate therapy.


```sparql
SELECT ?case
WHERE {
  ?case a type:Case .
  ?task a type:Task ;
        relation:partOf ?case ;
        relation:performedBy ?resource .
  ?resource a type:Resource .
  ?task relation:instanceOf Activity:IV%20Liquid .
  ?task relation:writes_value process_value:Age .
  ?task relation:writes_value process_value:DiagnosticBlood .
  ?task relation:writes_value process_value:DiagnosticLacticAcid .
  ?task relation:writes_value process_value:DisfuncOrg .
  ?task relation:writes_value process_value:Hypotensie .
  ?task relation:writes_value process_value:InfectionSuspected .
  FILTER (?pH <= 7.2) .
  FILTER (?AKIN_score >= 2 && ?AKIN_score <= 3) .
}
```

[]
For adults with septic shock and hypoperfusion-induced lactic acidemia, we suggest against using sodium bicarbonate therapy to improve hemodynamics or to reduce vasopressor requirement


```sparql
SELECT ?case
WHERE {
  ?case a type:Case .
  ?task a type:Task ;
        relation:partOf ?case ;
        relation:performedBy ?resource .
  ?activity a type:Activity ;
            relation:instanceOf ?task .
  ?activity relation:writes_value sepon:Lactate ;
            relation:writes_value sepon:Hypoxia .
  FILTER NOT EXISTS {
    ?activity relation:writes_value process_value:IV%20Liquid .
  }
}
```

[]
For adults with sepsis or septic shock, we suggest against using IV vitamin C.


...

[]
1. between ER Sepsis Triage and IV Antibiotics should be less than 1 hour


```sparql
SELECT ?case
WHERE {
  ?task1 a type:Task ;
          relation:partOf ?case ;
          instanceOf Activity:ER%20Sepsis%20Triage ;
          relation:completedAt ?time1 .
  ?task2 a type:Task ;
          relation:partOf ?case ;
          instanceOf Activity:IV%20Antibiotics ;
          relation:completedAt ?time2 .
  FILTER(?time2 - ?time1 >= xsd:duration("PT1H"))
}
```

[]
2. between ER Sepsis Triage and LacticAcid should be less than 3 hours.


```sparql
SELECT ?case
WHERE {
  ?task1 a type:Task ;
         relation:instanceOf Activity:ER%20Sepsis%20Triage ;
         relation:partOf ?case ;
         relation:completedAt ?time1 .
  ?task2 a type:Task ;
         relation:instanceOf Activity:LacticAcid ;
         relation:partOf ?case ;
         relation:completedAt ?time2 .
  FILTER(?time2 - ?time1 >= "PT3H"^^xsd:duration)
}
```

[]


# Getting it all in some kind of UI

In [100]:
import ipywidgets
from IPython.display import display

In [101]:
text = ipywidgets.Text(description='What\'s your name?')
button = ipywidgets.Button(description='Rename', on_click=lambda b: print(text.value))
output = ipywidgets.Output()

def on_button_click(b):
    with output:
        print(text.value)

button.on_click(on_button_click)
display(text, button, output)

Text(value='', description="What's your name?")

Button(description='Rename', style=ButtonStyle())

Output()

# Lab

In [170]:
prompt2 = ChatPromptTemplate.from_messages(
    [
        (
            "system",
'''You are a knowledge importer for a knowledge-graph-based business process management system.
Your task is to input textual process knowledge and an existing knowledge graph and output new nodes and edges representing the knowledge of the text.
Reuse existing owl:classes and owl:properties where appropriate. 

Please output the nodes and edges in RDF-turtle-syntax.
Output nothing else. Don't output any notes or justifications.

For generating the nodes and edges, please consider the following contextual schema information in OWL-RDF format and key entities in RDF format.  

{context}''',
        ),
        ("human", "{rule}"),
    ]
)
chain2 = prompt2 | llm

In [171]:
text = 'Draw at least two sets of blood cultures (aerobic and anaerobic) and cultures from suspected sources (e.g., urine, sputum, wound) before antibiotic administration. However, never delay antibiotics for culture collection beyond the initial hour.'
text = 'The activity CRP describes that a blood test for measuring CRP values is performed'
# text = 'Leucocytes, Lactic acid and CRP are all measurements provided by blood tests. Leucocytes are white blood cells and a high white blood count can signal an infection the body may be fighting [7]. CRP, or C-reactive protein can signal inflammation in the liver if the test result is high [8].'
response = chain2.invoke(
    {
        "context": prompt_context,
        "rule": text,
    }
).content
printmd(response)

```turtle
Activity:CRP a type:Activity ;
    rdfs:comment "A blood test for measuring CRP values is performed." .
```

'Activity:CRP a type:Activity ;\n    rdfs:comment "A blood test for measuring CRP values is performed." .'

In [176]:
print(f'{namespace_string(pkg)}\n\n{unwrap_markdown_code(response)}')

@prefix Activity: <http://example.org/instances/Activitys/> .
@prefix Diagnose: <http://example.org/instances/Diagnoses/> .
@prefix lifecycle%3Atransition: <http://example.org/instances/lifecycle%3Atransitions/> .
@prefix ns1: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#> .
@prefix org%3Agroup: <http://example.org/instances/org%3Agroups/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix process_value: <http://example.org/instances/process_values/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

Activity:CRP a type:Activity ;
    rdfs:comment "A blood test for measuring CRP values is performed." .


In [197]:
from itertools import zip_longest

{'a': '#999900', 'b': '#999900', 'c': '#999900'}

In [192]:
dict(zip(addition.all_nodes() - pkg.all_nodes(), ['#999900']))

{rdflib.term.Literal('A blood test for measuring CRP values is performed.'): '#999900'}

In [213]:
pkg -= list(Graph().parse(data=f'{namespace_string(pkg)}\n\n{unwrap_markdown_code(response)}', format='turtle'))[1:]

In [217]:
addition = Graph().parse(data=f'{namespace_string(pkg)}\n\n{addition_code}', format='turtle')
addition -= pkg
draw_graph(addition, color_func=lambda _: dict(zip_longest(addition.all_nodes() - pkg.all_nodes(), [], fillvalue='#99AA00')))
text = ipywidgets.Textarea(description='Current Code', value=addition_code)
button = ipywidgets.Button(description='Reload', on_click=lambda b: print(text.value))
output = ipywidgets.Output()

def on_button_click(b):
    pass

button.on_click(on_button_click)

GraphWidget(layout=Layout(height='500px', width='100%'))

In [203]:
pkg += addition

In [235]:
from rdflib.namespace import Namespace, DefinedNamespace
from rdflib.term import URIRef

BASE_URL = 'http://infs.cit.tum.de/karibdis/base/'

class BASE_PROCESS_ONTOLOGY(DefinedNamespace):

    _fail = True

    relation : URIRef
    
    _NS = Namespace(BASE_URL)

In [236]:
BASE_PROCESS_ONTOLOGY.relation

rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/base/relation')